# Coding assignment 1: Comparative Analysis of ML Classifiers for Medical Diagnosis
# ID 2671507

## Section A: Data Engineering (Pandas)

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the Breast Cancer Wisconsin dataset
cancer = load_breast_cancer()

# Convert the raw Bunch object into a Pandas DataFrame
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
# Checking for missing values
missing_values = df.isnull().sum()

# Display columns with missing values
if missing_values.sum() == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing values per column:")
    print(missing_values[missing_values > 0])

### Feature Scaling: Apply StandardScaler.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to the features
X_scaled = scaler.fit_transform(X)

# Convert scaled features back to a DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled successfully. Displaying the first 5 rows of scaled features:")
display(X_scaled_df.head())

### Correlation Matrix: Use Pandas to find the top 5 features most correlated with the target variable.

In [ ]:
# Combine scaled features and target for correlation calculation
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled['target'] = y

# Calculate the correlation matrix
correlation_matrix = df_scaled.corr()

# Get absolute correlations with the target variable and sort them in descending order
abs_target_correlations = correlation_matrix['target'].abs().sort_values(ascending=False)

# Display the top 5 features most correlated with the target variable (excluding the target)
top_5_correlated_features = abs_target_correlations.drop('target').head(5)

print("Top 5 features most correlated with the target variable (by absolute value):")
display(top_5_correlated_features)

## Section B: Model Implementation (Scikit-Learn)

#### Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# Split the scaled data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

#### 1. Logistic Regression (Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Initialize and train the Logistic Regression model
log_reg_model = LogisticRegression(random_state=42, solver='liblinear')
log_reg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_log_reg = log_reg_model.predict(X_test)

# Evaluate the model
accuracy_log_reg = accuracy_score(y_test, y_pred_log_reg)
precision_log_reg = precision_score(y_test, y_pred_log_reg)
recall_log_reg = recall_score(y_test, y_pred_log_reg)

print(f"Logistic Regression Accuracy: {accuracy_log_reg:.4f}")
print(f"Logistic Regression Precision: {precision_log_reg:.4f}")
print(f"Logistic Regression Recall: {recall_log_reg:.4f}")

#### 2. Random Forest Classifier (Ensemble method)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the Random Forest Classifier model
random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train, y_train)

# Make prediction on the test set
y_pred_random_forest = random_forest_model.predict(X_test)

# Evaluate the model
accuracy_random_forest = accuracy_score(y_test, y_pred_random_forest)
precision_random_forest = precision_score(y_test, y_pred_random_forest)
recall_random_forest = recall_score(y_test, y_pred_random_forest)

print(f"Random Forest Accuracy: {accuracy_random_forest:.4f}")
print(f"Random Forest Precision: {precision_random_forest:.4f}")
print(f"Random Forest Recall: {recall_random_forest:.4f}")

#### 3. Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

# Initialize and train the Support Vector Machine model
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)

# Make prediction on the test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate the model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)

print(f"SVM Accuracy: {accuracy_svm:.4f}")
print(f"SVM Precision: {precision_svm:.4f}")
print(f"SVM Recall: {recall_svm:.4f}")

# Store metrics for all models
model_metrics = {
    'Logistic Regression': {
        'Accuracy': accuracy_log_reg,
        'Precision': precision_log_reg,
        'Recall': recall_log_reg
    },
    'Random Forest': {
        'Accuracy': accuracy_random_forest,
        'Precision': precision_random_forest,
        'Recall': recall_random_forest
    },
    'SVM': {
        'Accuracy': accuracy_svm,
        'Precision': precision_svm,
        'Recall': recall_svm
    }
}

print("\nModel metrics stored successfully.")

## Section C: Visualization (Matplotlib)

#### Model Comparison: A bar chart comparing the Accuracy, Precision, and Recall of all three models.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare data for plotting
models = list(model_metrics.keys())
accuracy_scores = [model_metrics[model]['Accuracy'] for model in models]
precision_scores = [model_metrics[model]['Precision'] for model in models]
recall_scores = [model_metrics[model]['Recall'] for model in models]

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width, accuracy_scores, width, label='Accuracy')
rects2 = ax.bar(x, precision_scores, width, label='Precision')
rects3 = ax.bar(x + width, recall_scores, width, label='Recall')

ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy, Precision, and Recall')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.8, 1.0)

fig.tight_layout()
plt.show()

#### Confusion Matrix: A heatmap for the best performing model.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# The best performing model based on accuracy
best_model_name = max(model_metrics, key=lambda model: model_metrics[model]['Accuracy'])

# Get the predictions for the best model
if best_model_name == 'Logistic Regression':
    y_pred_best = y_pred_log_reg
elif best_model_name == 'Random Forest':
    y_pred_best = y_pred_random_forest
elif best_model_name == 'SVM':
    y_pred_best = y_pred_svm

# Generate the confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

# Plot the confusion matrix as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix for {best_model_name}')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


#### ROC Curve: For at least one model.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Get probability predictions for Logistic Regression
y_prob_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Calculate ROC curve values
fpr, tpr, thresholds = roc_curve(y_test, y_prob_log_reg)
roc_auc = roc_auc_score(y_test, y_prob_log_reg)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve - Logistic Regression')
plt.legend(loc='lower right')
plt.show()